# 构建一个函数来计算response整体的probs

In [4]:
import json
import os
from collections import Counter
import collections
from utils import save_jsonl, load_json, save_json, extract_entities, evaluate_sent, compute_f1, analyze_ner_errors, get_present_ner_error_types
import random
from typing import List, Tuple, Dict, Counter as CounterType

In [10]:
import re
import math
from typing import List, Dict, Any, Tuple, Set
import json # For example printing

def generate_error_demos(
    llm_entities_list: List[List[str]],
    gold_entities_list: List[List[str]]
) -> Dict[str, list]:
    """
    Compares LLM annotation list to Gold standard list (without offsets)
    to identify specific instances of Type, Span, Missing, and Spurious errors
    for generating few-shot examples.

    Args:
        llm_entities_list: List of [text, type] from LLM annotation.
                           e.g., [['perceptron', 'algorithm'], ['Perceptrons', 'product']]
        gold_entities_list: List of [text, type] from Gold standard.
                            e.g., [['perceptron models', 'algorithm'], ['Perceptrons', 'miscellaneous']]

    Returns:
        A dictionary containing lists of identified errors:
        {
            'type': [ [llm_text, llm_type, gold_text, gold_type], ... ],
            'span': [ [llm_text, llm_type, gold_text, gold_type], ... ],
            'missing': [ [gold_text, gold_type], ... ],
            'spurious': [ [llm_text, llm_type], ... ]
        }
    """
    # --- 1. Pre-process Inputs ---
    # Use tuples {(text, type)} for efficient set operations
    llm_set: Set[Tuple[str, str]] = {tuple(item) for item in llm_entities_list}
    gold_set: Set[Tuple[str, str]] = {tuple(item) for item in gold_entities_list}

    # Create maps for text -> type lookup.
    # NOTE: This simple dict assumes unique text strings. If the same text can appear
    # with different types *within the same list*, this might need adjustment.
    # For this specific task comparing LLM vs Gold, it's usually sufficient.
    llm_map: Dict[str, str] = {text: type_ for text, type_ in llm_entities_list}
    gold_map: Dict[str, str] = {text: type_ for text, type_ in gold_entities_list}

    # --- 2. Initialize Results & Tracking ---
    type_errors: List[List[str]] = []
    span_errors: List[List[str]] = []
    missing: List[List[str]] = []
    spurious: List[List[str]] = []

    # Keep track of tuples that have been definitively matched (Exact, Type, Span)
    # This prevents double counting errors or misclassifying parts of pairs.
    matched_llm_tuples: Set[Tuple[str, str]] = set()
    matched_gold_tuples: Set[Tuple[str, str]] = set()

    # --- 3. Pass 1: Exact Matches (Text + Type) ---
    # Find entities present in both sets
    exact_matches = llm_set.intersection(gold_set)
    matched_llm_tuples.update(exact_matches)
    matched_gold_tuples.update(exact_matches)
    # print(f"Exact Matches: {exact_matches}") # Debugging

    # --- 4. Pass 2: Type Errors (Same Text, Different Type) ---
    # Iterate through LLM entities not already exactly matched
    for llm_text, llm_type in llm_set:
        llm_tuple = (llm_text, llm_type)
        if llm_tuple in matched_llm_tuples:
            continue

        # Check if the same text exists in gold standard with a different type
        if llm_text in gold_map:
            gold_type = gold_map[llm_text]
            if llm_type != gold_type:
                gold_tuple = (llm_text, gold_type)
                # Ensure this corresponding gold entity actually exists in the gold set
                # and hasn't already been matched (e.g., exactly matched with a different LLM entity if duplicates existed)
                if gold_tuple in gold_set and gold_tuple not in matched_gold_tuples:
                    type_errors.append([llm_text, llm_type, llm_text, gold_type])
                    matched_llm_tuples.add(llm_tuple)
                    matched_gold_tuples.add(gold_tuple)
                    # print(f"Type Error Found: {llm_tuple} vs {gold_tuple}") # Debugging


    # --- 5. Pass 3: Span Errors (Substring Overlap Heuristic) ---
    # Iterate through remaining unmatched LLM entities
    for llm_text, llm_type in llm_set:
        llm_tuple = (llm_text, llm_type)
        if llm_tuple in matched_llm_tuples:
            continue

        # Compare against remaining unmatched Gold entities
        for gold_text, gold_type in gold_set:
            gold_tuple = (gold_text, gold_type)
            if gold_tuple in matched_gold_tuples:
                continue

            # Check for non-identical text AND substring relationship
            # Ensure texts are not empty before checking 'in'
            if llm_text and gold_text and llm_text != gold_text and \
               (llm_text in gold_text or gold_text in llm_text):
                # Found a potential span error match
                span_errors.append([llm_text, llm_type, gold_text, gold_type])
                matched_llm_tuples.add(llm_tuple)
                matched_gold_tuples.add(gold_tuple)
                # print(f"Span Error Found: {llm_tuple} vs {gold_tuple}") # Debugging
                # Break inner loop: Match this LLM entity to the first overlapping Gold entity found
                # This prevents one LLM entity matching multiple Gold entities as span errors.
                break

    # --- 6. Pass 4: Missing and Spurious ---
    # Missing: Gold entities not matched in any previous pass
    for gold_text, gold_type in gold_set:
        gold_tuple = (gold_text, gold_type)
        if gold_tuple not in matched_gold_tuples:
            missing.append([gold_text, gold_type])
            # print(f"Missing Found: {gold_tuple}") # Debugging


    # Spurious: LLM entities not matched in any previous pass
    for llm_text, llm_type in llm_set:
        llm_tuple = (llm_text, llm_type)
        if llm_tuple not in matched_llm_tuples:
            spurious.append([llm_text, llm_type])
            # print(f"Spurious Found: {llm_tuple}") # Debugging


    # --- 7. Return Result Dictionary ---
    return {
        'type': type_errors,
        'span': span_errors,
        'missing': missing,
        'spurious': spurious
    }

def compute_entity_tag_entropies(
    seq_logits: List[Dict[str, Any]],
    per_token: bool = False,
    bits: bool = False
) -> Tuple[List[float], float]:
    """
    计算每个实体标签的熵，以及它们的平均熵。

    熵的定义：H = - sum_i log P(token_i | context_i)
    如果 per_token=True，则返回平均每个 token 的熵：H/n
    如果 bits=True，则将自然对数换算成以 2 为底： H_bits = H / ln(2)

    Args:
        seq_logits: 按生成顺序的 token 列表，每项是 dict，包含
                    - 'token': str
                    - 'logprob': float
        per_token:  是否输出“每个 token 的平均熵”（默认 False）
        bits:       是否将熵转换为比特（bits）单位（默认 False）

    Returns:
        entity_entropies: 各实体标签的熵列表
        avg_entropy:      全部实体标签熵的算术平均
    """
    # 1. 重建全文本并记录每个 token 的字符跨度
    full_text = ""
    spans: List[Tuple[int, int]] = []
    for tok in seq_logits:
        start = len(full_text)
        full_text += tok['token']
        end = len(full_text)
        spans.append((start, end))

    # 2. 正则匹配所有实体标签
    pattern = re.compile(r'<entity[^>]*>.*?</entity>')
    matches = list(pattern.finditer(full_text))
    if not matches:
        raise ValueError("在生成文本中未找到任何 <entity> 标签。")

    # 3. 对每个实体标签，累加其对应 token 的负 logprob（熵）
    entropies: List[float] = []
    for m in matches:
        s_char, e_char = m.span()
        # 找到覆盖该标签的 token 索引
        idxs = [
            i for i, (s, e) in enumerate(spans)
            if not (e <= s_char or s >= e_char)
        ]
        # 累加 -logprob
        H = -sum(seq_logits[i]['logprob'] for i in idxs)
        # 如需平均每 token 熵：
        if per_token and idxs:
            H = H / len(idxs)
        # 如需转换成 bits：
        if bits:
            H = H / math.log(2)
        entropies.append(H)

    # 4. 计算平均熵
    avg_H = sum(entropies) / len(entropies)
    return entropies, avg_H

def compute_entity_tag_probs(
    seq_logits: List[Dict[str, Any]]
) -> Tuple[List[float], float]:
    """
    计算每个实体标签的概率以及它们的平均概率。

    Args:
        seq_logits: 按生成顺序的 token 列表，每项是 dict，必须包含
                    - 'token': str
                    - 'logprob': float

    Returns:
        entity_probs: 各实体标签的概率列表
        avg_prob:       全部实体标签概率的算术平均
    """
    # 1. 重建全文本并记录每个 token 的字符跨度
    full_text = ""
    spans: List[Tuple[int, int]] = []
    for tok in seq_logits:
        start = len(full_text)
        full_text += tok['token']
        end = len(full_text)
        spans.append((start, end))

    # 2. 正则匹配所有实体标签
    pattern = re.compile(r'<entity[^>]*>.*?</entity>')
    matches = list(pattern.finditer(full_text))
    if not matches:
        raise ValueError("在生成文本中未找到任何 <entity> 标签。")

    # 3. 针对每个实体标签，累加其对应 token 的 logprob 并转为概率
    entity_probs: List[float] = []
    for m in matches:
        s_char, e_char = m.span()
        # 收集所有与该标签字符范围有重叠的 token 索引
        idxs = [
            i for i, (s, e) in enumerate(spans)
            if not (e <= s_char or s >= e_char)
        ]
        # 累加 logprob
        logp_sum = sum(seq_logits[i]['logprob'] for i in idxs)
        # 转为概率
        prob = math.exp(logp_sum)
        entity_probs.append(prob)

    # 4. 计算平均
    avg_prob = sum(entity_probs) / len(entity_probs)
    return entity_probs, avg_prob


def compute_sequence_entropy(
    seq_logits: List[Dict[str, Any]],
    per_token: bool = False,
    bits: bool = False
) -> Tuple[float, float]:
    """
    计算整个序列的熵，以及可选的每 token 平均熵。

    熵定义：H = -∑ log P(token_i | context_i)
    参数:
        seq_logits: 按生成顺序的 token 列表，每项包含:
            - 'token': str
            - 'logprob': float
        per_token: 是否返回平均每 token 熵 (H/n)
        bits:      是否将熵转换为以 2 为底的比特 (bits)

    返回:
        total_entropy: 整个序列的熵 (nats 或 bits)
        avg_entropy:   每 token 平均熵 (如果 per_token=True), 否则为None
    """
    # 计算总熵 (nats)
    total_entropy = -sum(item['logprob'] for item in seq_logits)

    # 计算平均熵（如果需要）
    avg_entropy = None
    if per_token and seq_logits:
        avg_entropy = total_entropy / len(seq_logits)

    # 转换为 bits 单位（如果需要）
    if bits:
        total_entropy /= math.log(2)
        if avg_entropy is not None:
            avg_entropy /= math.log(2)

    return total_entropy, avg_entropy

def get_ner_scores(eva_data, col = 'response'):
    counts = Counter()
    for sent in eva_data:
        if col not in sent:
            continue
        pred = extract_entities(sent[col])
        ref = sent['ners']
        counts = evaluate_sent(ref, pred, counts)
    scores_ner = compute_f1(counts["ner_predicted"], counts["ner_gold"], counts["ner_matched"])
    return scores_ner


# ——— 用法示例 ———
# seq_logits = data[i]['logits']
# entity_probs, avg_prob = compute_entity_tag_probs(seq_logits)
# print("每个实体标签的概率：", entity_probs)
# print("平均实体标签概率：", avg_prob)

# # 计算熵
# entity_entropies, avg_entropy = compute_entity_tag_entropies(seq_logits)
# print("每个实体标签的熵：", entity_entropies)
# print("平均实体标签熵：", avg_entropy)

In [11]:
data_path = "./output/aug/ai/Qwen2.5-72B-demo100-ann-dev-logits.json"
data = load_json(data_path)
counts = Counter()
for sent in data:
    if "response" not in sent:
        continue
    sent['response'] = sent['response'].replace("Output: ", "")
    pred = extract_entities(sent['response'])
    ref = sent['ners']
    counts = evaluate_sent(ref, pred, counts)
scores_ner = compute_f1(counts["ner_predicted"], counts["ner_gold"], counts["ner_matched"])
# print(f"======= {i} =======")
print(scores_ner)

{'precision': 0.7357414448669202, 'recall': 0.749515816655907, 'f1': 0.7425647585545252}


In [12]:
# 处理数据
no_errors = 0
errors_stat = collections.defaultdict(int)
for sent in data:
    sent['response'] = sent['response'].replace("Output: ", "")
    pred = extract_entities(sent['response'])
    ref = sent['ners']
    errors = generate_error_demos(pred, ref)
    sent['errors'] = errors
    if len(errors['type'] + errors['span'] + errors['missing'] + errors['spurious']) == 0:
        # print("No errors found.")
        no_errors += 1
        sent['correct'] = True
    else:
        sent['correct'] = False
    # statistics
    for error_type, error_list in errors.items():
        if error_type == 'type':
            errors_stat['type'] += len(error_list)
        elif error_type == 'span':
            errors_stat['span'] += len(error_list)
        elif error_type == 'missing':
            errors_stat['missing'] += len(error_list)
        elif error_type == 'spurious':
            errors_stat['spurious'] += len(error_list)
print(f"Total: {len(data)}, No errors: {no_errors}, Errors: {len(data) - no_errors}")
print("Errors statistics:")
for error_type, count in errors_stat.items():
    print(f"{error_type}: {count}")
# save 
# save_json(data, "./output/aug/ai/deepseek-chat-demo100-ann-dev-logits-errors.json")

Total: 350, No errors: 134, Errors: 216
Errors statistics:
type: 183
span: 98
missing: 102
spurious: 129


In [13]:
# 计算每个样本的熵和概率
for i in range(len(data)):
    seq_logits = data[i]['logits']
    entity_probs, avg_prob = compute_entity_tag_probs(seq_logits)
    entity_entropies, avg_entropy = compute_entity_tag_entropies(seq_logits)
    data[i]['entity_probs'] = entity_probs
    data[i]['avg_prob'] = avg_prob
    data[i]['entity_entropies'] = entity_entropies
    data[i]['avg_entropy'] = avg_entropy
    # 计算整个句子的熵, compute_sequence_entropy
    data[i]['total_entropy'], data[i]['avg_entropy'] = compute_sequence_entropy(seq_logits, per_token=True, bits=True)

In [15]:
# 用avg_prob对所有数据进行排序
data.sort(key=lambda x: x['avg_entropy'], reverse=False)
# 前33%有多少是sent['correct'] = True
# 取前33%的数据
top_N = int(len(data) * 0.33)
data_top_N = data[:top_N]
correct_count = 0
for sent in data_top_N:
    if sent['correct']:
        correct_count += 1
print(f"Top 33% correct count: {correct_count}, Top 33% total count: {top_N}")
# 输出错误率
error_rate = (top_N - correct_count) / top_N
print(f"Top N% error rate: {error_rate}")
print(get_ner_scores(data_top_N, col = 'response'))

Top 33% correct count: 77, Top 33% total count: 115
Top N% error rate: 0.33043478260869563
{'precision': 0.8734177215189873, 'recall': 0.8808510638297873, 'f1': 0.8771186440677966}


In [16]:
# 用avg_prob对所有数据进行排序
data.sort(key=lambda x: x['total_entropy'], reverse=False)
# 前33%有多少是sent['correct'] = True
# 取前33%的数据
top_N = int(len(data) * 0.33)
data_top_N = data[:top_N]
correct_count = 0
for sent in data_top_N:
    if sent['correct']:
        correct_count += 1
print(f"Top 33% correct count: {correct_count}, Top 33% total count: {top_N}")
# 输出错误率
error_rate = (top_N - correct_count) / top_N
print(f"Top N% error rate: {error_rate}")
print(get_ner_scores(data_top_N, col = 'response'))

Top 33% correct count: 81, Top 33% total count: 115
Top N% error rate: 0.2956521739130435
{'precision': 0.888631090487239, 'recall': 0.9011764705882352, 'f1': 0.8948598130841121}


In [17]:
# 用avg_prob对所有数据进行排序
data.sort(key=lambda x: x['avg_prob'], reverse=True)
print("Entire data set scores:")
print(get_ner_scores(data, col = 'response'))
rates = [0.25, 0.33, 0.5, 0.66, 0.75, 0.8, 0.9]
for rate in rates:
    print(f"================= Top {rate*100}% Confident Samples =================")
    top_N = int(len(data) * rate)
    data_top_N = data[:top_N]
    correct_count = 0
    for sent in data_top_N:
        if sent['correct']:
            correct_count += 1
    print(f"Top {rate*100}% correct count: {correct_count}, Top {rate*100}%% total count: {top_N}")
    # 输出错误率
    error_rate = (top_N - correct_count) / top_N
    print(f"Top {rate*100}%% error rate: {error_rate}")
    print(get_ner_scores(data_top_N, col = 'response'))

Entire data set scores:
{'precision': 0.7357414448669202, 'recall': 0.749515816655907, 'f1': 0.7425647585545252}
================= Top 25.0% Confident Samples =================
Top 25.0% correct count: 62, Top 25.0%% total count: 87
Top 25.0%% error rate: 0.28735632183908044
{'precision': 0.9315068493150684, 'recall': 0.8888888888888888, 'f1': 0.9096989966555183}
================= Top 33.0% Confident Samples =================
Top 33.0% correct count: 76, Top 33.0%% total count: 115
Top 33.0%% error rate: 0.3391304347826087
{'precision': 0.8888888888888888, 'recall': 0.8769574944071589, 'f1': 0.8828828828828829}
================= Top 50.0% Confident Samples =================
Top 50.0% correct count: 98, Top 50.0%% total count: 175
Top 50.0%% error rate: 0.44
{'precision': 0.8457374830852503, 'recall': 0.8434547908232118, 'f1': 0.8445945945945945}
================= Top 66.0% Confident Samples =================
Top 66.0% correct count: 110, Top 66.0%% total count: 231
Top 66.0%% error rat

In [40]:
# 用avg_prob对所有数据进行排序
data.sort(key=lambda x: x['entity_probs'], reverse=True)
# 前33%有多少是sent['correct'] = True
# 取前33%的数据
top_N = int(len(data) * 0.25)
data_top_N = data[:top_N]
correct_count = 0
for sent in data_top_N:
    if sent['correct']:
        correct_count += 1
print(f"Top 33% correct count: {correct_count}, Top 33% total count: {top_N}")
# 输出错误率
error_rate = (top_N - correct_count) / top_N
print(f"Top 33% error rate: {error_rate}")
print(get_ner_scores(data_top_N, col = 'response'))

Top 33% correct count: 47, Top 33% total count: 87
Top 33% error rate: 0.45977011494252873
{'precision': 0.8756476683937824, 'recall': 0.8825065274151436, 'f1': 0.8790637191157347}


In [27]:
# 用avg_prob对所有数据进行排序
data.sort(key=lambda x: x['total_entropy'], reverse=False)
# 前33%有多少是sent['correct'] = True
# 取前33%的数据
top_N = int(len(data) * 0.33)
data_top_N = data[:top_N]
correct_count = 0
for sent in data_top_N:
    if sent['correct']:
        correct_count += 1
print(f"Top 33% correct count: {correct_count}, Top 33% total count: {top_N}")
# 输出错误率
error_rate = (top_N - correct_count) / top_N
print(f"Top 33% error rate: {error_rate}")
print(get_ner_scores(data_top_N, col = 'response'))

Top 33% correct count: 72, Top 33% total count: 115
Top 33% error rate: 0.3739130434782609
{'precision': 0.8453159041394336, 'recall': 0.849015317286652, 'f1': 0.8471615720524017}


In [54]:
# # 打印错误的样本
# for sent in data_top_N:
#     if not sent['correct']:
#         print(f"Sample: {sent['response']}")
#         print(f"Errors: {sent['errors']}")
#         print(f"Entity probs: {sent['entity_probs']}")
#         print(f"Avg prob: {sent['avg_prob']}")
#         print(f"Entity entropies: {sent['entity_entropies']}")
#         print(f"Avg entropy: {sent['avg_entropy']}")
#         print("===" * 10)
# # 计算熵和概率

In [68]:
# 用avg_entropy对所有数据进行排序
data.sort(key=lambda x: x['avg_entropy'], reverse=False)
# 前33%有多少是sent['correct'] = True
# 取前33%的数据
top_N = int(len(data) * 0.33)
data_top_N = data[:top_N]
correct_count = 0
for sent in data_top_N:
    if sent['correct']:
        correct_count += 1
print(f"Top 33% correct count: {correct_count}, Top 33% total count: {top_N}")

Top 33% correct count: 74, Top 33% total count: 115


# 统计dev 和 test set中常见的type的错误
目标：经过prompt后可以减少30-40个typing error

**先分析test set** 结果

In [26]:
# output/aug/ai/deepseek-chat-demo100-ann-devtest-test-test-errors.json
import json
import os
from collections import Counter
import collections
from utils import save_jsonl, load_json, save_json, extract_entities, evaluate_sent, compute_f1, analyze_ner_errors, get_present_ner_error_types
import random
from typing import List, Tuple, Dict, Counter as CounterType

data = load_json("output/aug/ai/deepseek-chat-demo100-ann-devtest-test-test-errors.json")

In [ ]:
error_map_stat = {}
for item in data:
    type_errors = item['errors']['type']
    for error in type_errors:
        map_str = f"{error[3]}->{error[1]}"
        if map_str not in error_map_stat:
            error_map_stat[map_str] = []
        exp_str = f"Text: {item['text']}\n" + \
                  f"Entity: {error[0]}\n" \
                  f"Wrong Type: {error[1]}\n" +\
                  f"Correct Type: {error[3]}\n"
        error_map_stat[map_str].append(exp_str)
# print the number of each key
error_stat = {}
for key, value in error_map_stat.items():
    error_stat[key] = len(value)
# sort the dictionary by value
error_stat = dict(sorted(error_stat.items(), key=lambda item: item[1], reverse=True))
# print the top 10 keys
sum_error = 0
for key, value in list(error_stat.items()):
    print(f"{key}: {value}")
    sum_error += value
print(f"Total number of errors: {sum_error}")

In [ ]:
# ['country', 'location', 'metrics', 'organization', 'programming language', 'researcher', 'university', 'person']

In [30]:
# targets = ['country', 'location', 'metrics', 'organization', 'programming language', 'researcher', 'university', 'person']
# for target in targets:
#     print(f"错误的把 {target} 识别为其他类型:")
#     target_num = 0
#     for key, val in error_map_stat.items():
#         tps = key.split("->")
#         if tps[0] == target:
#             print(f"错误的把 {tps[0]} 识别为 {tps[1]}")
#             target_num += len(error_map_stat[key])
#             for val in error_map_stat[key]:
#                 print(val)
#     print("===" * 20)

# LLM Many-shot 标注错误分析
分析第一轮annotation的Error 类型和找到对应的合适的Prompt

In [1]:
import json
import os
from collections import Counter
import collections
from utils import save_jsonl, load_jsonl, load_json, save_json, extract_entities, evaluate_sent, compute_f1, analyze_ner_errors, get_present_ner_error_types
import random
from typing import List, Tuple, Dict, Counter as CounterType

**DeepSeek 100 Shots** AI Domain

In [8]:
# ./output/aug/ai/deepseek-chat-pos100-neg10-ann-all.json
data = load_json(f"./output/aug/ai/deepseek-chat-pos100-neg10-ann-all.json")
# 错误类型
print(data[0].keys())

dict_keys(['idx', 'text', 'ners', 'target', 'sent_id', 'str_words', 'tags_ner', 'response'])


In [9]:
data_samples: List[Dict[str, List[List[str]]]] = []
for i in range(len(data)):
    data[i]['response'] = data[i]['response'].replace("Output: ", "")
    data[i]['pred_ners'] = extract_entities(data[i]['response'])
    data_samples.append(
        {
            "llm_prediction": data[i]['pred_ners'],
            "ground_truth": data[i]['ners'],
        }
    )
# 统计每个样本的错误类型
# --- 2. Run Analysis ---
total_results: CounterType[str] = collections.Counter() # Aggregates counts across all samples
total_gt_entities: int = 0
total_llm_entities: int = 0

print(f"Starting NER error analysis on {len(data_samples)} samples...")

for i, sample in enumerate(data_samples):
    if 'llm_prediction' not in sample or 'ground_truth' not in sample:
        print(f"Warning: Sample {i+1} is missing 'llm_prediction' or 'ground_truth'. Skipping.")
        continue

    llm_pred = sample['llm_prediction']
    gt = sample['ground_truth']

    # Analyze the single sample
    try:
        sample_results = analyze_ner_errors(llm_pred, gt)
        # Update total results
        total_results.update(sample_results)
        # Update total entity counts
        total_gt_entities += len(gt)
        total_llm_entities += len(llm_pred)
    except Exception as e:
        print(f"Error processing sample {i+1}: {e}")
        # Optionally add more details about the sample causing the error

print("Analysis complete.")


# --- 3. Display Results ---
print("\n--- Overall NER Error Analysis ---")
print(f"Total Samples Analyzed: {len(data_samples)}")
print("-" * 35)
print(f"Total Ground Truth Entities:   {total_gt_entities}")
print(f"Total LLM Predicted Entities:  {total_llm_entities}")
print("-" * 35)
print("Entity Counts:")
print(f"  Correct Matches:      {total_results['correct']}")
print(f"  Entity Type Errors:   {total_results['type']}")
print(f"  Entity Span Errors:   {total_results['span']} (using substring matching)")
print(f"  Missing Entities:     {total_results['missing']}")
print(f"  Spurious Entities:    {total_results['spurious']}")
print("-" * 35)

# --- Performance Metrics (Strict - Type/Span count as FP & FN) ---
# True Positives (TP) = Correct Matches
# False Positives (FP) = Spurious Errors + Type Errors + Span Errors
# False Negatives (FN) = Missing Errors + Type Errors + Span Errors

tp = total_results['correct']
fp = total_results['spurious'] + total_results['type'] + total_results['span']
fn = total_results['missing'] + total_results['type'] + total_results['span']

# Avoid division by zero
precision = tp / (tp + fp) if (tp + fp) > 0 else 0
recall = tp / (tp + fn) if (tp + fn) > 0 else 0
f1_score = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

print("Performance Metrics (Strict Matching):")
print(f"  Precision: {precision:.4f}  (TP / (TP + FP))")
print(f"  Recall:    {recall:.4f}  (TP / (TP + FN))")
print(f"  F1-Score:  {f1_score:.4f}")
print("-" * 35)

Starting NER error analysis on 350 samples...
Analysis complete.

--- Overall NER Error Analysis ---
Total Samples Analyzed: 350
-----------------------------------
Total Ground Truth Entities:   1591
Total LLM Predicted Entities:  1676
-----------------------------------
Entity Counts:
  Correct Matches:      1252
  Entity Type Errors:   166
  Entity Span Errors:   106 (using substring matching)
  Missing Entities:     67
  Spurious Entities:    152
-----------------------------------
Performance Metrics (Strict Matching):
  Precision: 0.7470  (TP / (TP + FP))
  Recall:    0.7869  (TP / (TP + FN))
  F1-Score:  0.7665
-----------------------------------


In [2]:
# ./output/aug/ai/deepseek-chat-pos100-neg10-ann-all.json
data = load_json(f"./output/aug/ai/modifier/deepseek-chat-pos100-neg10-modifier-all-100.json")
# 错误类型
# randomly sample 100 samples
# random.seed(42)
# random.shuffle(data)
# data = data[:100]
print(data[0].keys())
# 

dict_keys(['idx', 'text', 'ners', 'target', 'sent_id', 'str_words', 'tags_ner', 'response', 'judge_status', 'judge', 'modifier_response'])


In [3]:
data_samples: List[Dict[str, List[List[str]]]] = []
for i in range(len(data)):
    if 'response' not in data[i]:
        continue
    data[i]['modifier_response'] = data[i]['modifier_response'].replace("**Final Annotated Text**:\n", "")
    data[i]['preds'] = extract_entities(data[i]['modifier_response'])
    data_samples.append(
        {
            "llm_prediction": data[i]['preds'],
            "ground_truth": data[i]['ners'],
        }
    )
# 统计每个样本的错误类型
# --- 2. Run Analysis ---
total_results: CounterType[str] = collections.Counter() # Aggregates counts across all samples
total_gt_entities: int = 0
total_llm_entities: int = 0

print(f"Starting NER error analysis on {len(data_samples)} samples...")

for i, sample in enumerate(data_samples):
    if 'llm_prediction' not in sample or 'ground_truth' not in sample:
        print(f"Warning: Sample {i+1} is missing 'llm_prediction' or 'ground_truth'. Skipping.")
        continue

    llm_pred = sample['llm_prediction']
    gt = sample['ground_truth']

    # Analyze the single sample
    try:
        sample_results = analyze_ner_errors(llm_pred, gt)
        # Update total results
        total_results.update(sample_results)
        # Update total entity counts
        total_gt_entities += len(gt)
        total_llm_entities += len(llm_pred)
    except Exception as e:
        print(f"Error processing sample {i+1}: {e}")
        # Optionally add more details about the sample causing the error

print("Analysis complete.")


# --- 3. Display Results ---
print("\n--- Overall NER Error Analysis ---")
print(f"Total Samples Analyzed: {len(data_samples)}")
print("-" * 35)
print(f"Total Ground Truth Entities:   {total_gt_entities}")
print(f"Total LLM Predicted Entities:  {total_llm_entities}")
print("-" * 35)
print("Entity Counts:")
print(f"  Correct Matches:      {total_results['correct']}")
print(f"  Entity Type Errors:   {total_results['type']}")
print(f"  Entity Span Errors:   {total_results['span']} (using substring matching)")
print(f"  Missing Entities:     {total_results['missing']}")
print(f"  Spurious Entities:    {total_results['spurious']}")
print("-" * 35)

# --- Performance Metrics (Strict - Type/Span count as FP & FN) ---
# True Positives (TP) = Correct Matches
# False Positives (FP) = Spurious Errors + Type Errors + Span Errors
# False Negatives (FN) = Missing Errors + Type Errors + Span Errors

tp = total_results['correct']
fp = total_results['spurious'] + total_results['type'] + total_results['span']
fn = total_results['missing'] + total_results['type'] + total_results['span']

# Avoid division by zero
precision = tp / (tp + fp) if (tp + fp) > 0 else 0
recall = tp / (tp + fn) if (tp + fn) > 0 else 0
f1_score = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

print("Performance Metrics (Strict Matching):")
print(f"  Precision: {precision:.4f}  (TP / (TP + FP))")
print(f"  Recall:    {recall:.4f}  (TP / (TP + FN))")
print(f"  F1-Score:  {f1_score:.4f}")
print("-" * 35)

Starting NER error analysis on 350 samples...
Analysis complete.

--- Overall NER Error Analysis ---
Total Samples Analyzed: 350
-----------------------------------
Total Ground Truth Entities:   1591
Total LLM Predicted Entities:  1610
-----------------------------------
Entity Counts:
  Correct Matches:      1132
  Entity Type Errors:   205
  Entity Span Errors:   117 (using substring matching)
  Missing Entities:     137
  Spurious Entities:    156
-----------------------------------
Performance Metrics (Strict Matching):
  Precision: 0.7031  (TP / (TP + FP))
  Recall:    0.7115  (TP / (TP + FN))
  F1-Score:  0.7073
-----------------------------------


In [7]:
data_samples: List[Dict[str, List[List[str]]]] = []
for i in range(len(data)):
    if 'modifier_response' not in data[i]:
        continue
    data[i]['modifier_response'] = data[i]['modifier_response'].replace("**Final Annotated Text**:\n", "")
    data[i]['modifier_ners'] = extract_entities(data[i]['modifier_response'])
    data_samples.append(
        {
            "llm_prediction": data[i]['modifier_ners'],
            "ground_truth": data[i]['ners'],
        }
    )
# 统计每个样本的错误类型
# --- 2. Run Analysis ---
total_results: CounterType[str] = collections.Counter() # Aggregates counts across all samples
total_gt_entities: int = 0
total_llm_entities: int = 0

print(f"Starting NER error analysis on {len(data_samples)} samples...")

for i, sample in enumerate(data_samples):
    if 'llm_prediction' not in sample or 'ground_truth' not in sample:
        print(f"Warning: Sample {i+1} is missing 'llm_prediction' or 'ground_truth'. Skipping.")
        continue

    llm_pred = sample['llm_prediction']
    gt = sample['ground_truth']

    # Analyze the single sample
    try:
        sample_results = analyze_ner_errors(llm_pred, gt)
        # Update total results
        total_results.update(sample_results)
        # Update total entity counts
        total_gt_entities += len(gt)
        total_llm_entities += len(llm_pred)
    except Exception as e:
        print(f"Error processing sample {i+1}: {e}")
        # Optionally add more details about the sample causing the error

print("Analysis complete.")


# --- 3. Display Results ---
print("\n--- Overall NER Error Analysis ---")
print(f"Total Samples Analyzed: {len(data_samples)}")
print("-" * 35)
print(f"Total Ground Truth Entities:   {total_gt_entities}")
print(f"Total LLM Predicted Entities:  {total_llm_entities}")
print("-" * 35)
print("Entity Counts:")
print(f"  Correct Matches:      {total_results['correct']}")
print(f"  Entity Type Errors:   {total_results['type']}")
print(f"  Entity Span Errors:   {total_results['span']} (using substring matching)")
print(f"  Missing Entities:     {total_results['missing']}")
print(f"  Spurious Entities:    {total_results['spurious']}")
print("-" * 35)

# --- Performance Metrics (Strict - Type/Span count as FP & FN) ---
# True Positives (TP) = Correct Matches
# False Positives (FP) = Spurious Errors + Type Errors + Span Errors
# False Negatives (FN) = Missing Errors + Type Errors + Span Errors

tp = total_results['correct']
fp = total_results['spurious'] + total_results['type'] + total_results['span']
fn = total_results['missing'] + total_results['type'] + total_results['span']

# Avoid division by zero
precision = tp / (tp + fp) if (tp + fp) > 0 else 0
recall = tp / (tp + fn) if (tp + fn) > 0 else 0
f1_score = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

print("Performance Metrics (Strict Matching):")
print(f"  Precision: {precision:.4f}  (TP / (TP + FP))")
print(f"  Recall:    {recall:.4f}  (TP / (TP + FN))")
print(f"  F1-Score:  {f1_score:.4f}")
print("-" * 35)

Starting NER error analysis on 328 samples...
Analysis complete.

--- Overall NER Error Analysis ---
Total Samples Analyzed: 328
-----------------------------------
Total Ground Truth Entities:   1488
Total LLM Predicted Entities:  1507
-----------------------------------
Entity Counts:
  Correct Matches:      1055
  Entity Type Errors:   191
  Entity Span Errors:   111 (using substring matching)
  Missing Entities:     131
  Spurious Entities:    150
-----------------------------------
Performance Metrics (Strict Matching):
  Precision: 0.7001  (TP / (TP + FP))
  Recall:    0.7090  (TP / (TP + FN))
  F1-Score:  0.7045
-----------------------------------


**对错误类型进行归类**

In [1]:
import json
import os
from collections import Counter
import collections
from utils import save_jsonl, load_jsonl, load_json, save_json, extract_entities, evaluate_sent, compute_f1, analyze_ner_errors, get_present_ner_error_types
import random
from typing import List, Tuple, Dict, Counter as CounterType
data = load_json(f"./datasets/aug/ai/negatives.json")
# 错误类型
print(data[0].keys())

dict_keys(['text', 'target', 'entities', 'response'])


In [4]:
error_sample_counter = {
    "correct": 0,
    "type": 0,
    "span": 0,
    "missing": 0,
    "spurious": 0
}
data_samples: List[Dict[str, List[List[str]]]] = []
for i in range(len(data)):
    data[i]['response'] = data[i]['response'].replace("Output: ", "")
    data[i]['pred_ners'] = extract_entities(data[i]['response'])
    llm_prediction = data[i]['pred_ners']
    ground_truth = data[i]['entities']
    # 判断该样本的错误类型有哪些
    error_types = get_present_ner_error_types(llm_prediction, ground_truth)
    print(f"===== Sample {i+1} =====")
    print(f"Sample {i+1} Error Types: {error_types}")
    print(data[i]['target'])
    print(data[i]['response'])
    # update the error sample counter
    if "correct" in error_types:
        error_sample_counter["correct"] += 1
    if "type" in error_types:
        error_sample_counter["type"] += 1
    if "span" in error_types:
        error_sample_counter["span"] += 1
    if "missing" in error_types:
        error_sample_counter["missing"] += 1
    if "spurious" in error_types:
        error_sample_counter["spurious"] += 1

===== Sample 1 =====
Sample 1 Error Types: {'correct', 'span', 'type'}
<entity type="product">GATE</entity> includes an <entity type="task">information extraction</entity> system called <entity type="product">ANNIE</entity> ( <entity type="product">A Nearly-New Information Extraction System</entity> ) which is a set of modules comprising a <entity type="miscellaneous">tokenizer</entity> , a <entity type="miscellaneous">gazetteer</entity> , a <entity type="miscellaneous">sentence splitter</entity> , a <entity type="task">Part-of-speech tagging</entity> , a <entity type="product">Named entity recognition transducer</entity> and a <entity type="product">coreference tagger</entity> .
<entity type="product">GATE</entity> includes an <entity type="task">information extraction</entity> system called <entity type="product">ANNIE</entity> ( <entity type="product">A Nearly-New Information Extraction System</entity> ) which is a set of modules comprising a <entity type="algorithm">tokenizer</enti

In [5]:
error_sample_counter

{'correct': 27, 'type': 16, 'span': 13, 'missing': 6, 'spurious': 13}

In [6]:
# 统计每个样本的错误类型
# --- 2. Run Analysis ---
total_results: CounterType[str] = collections.Counter() # Aggregates counts across all samples
total_gt_entities: int = 0
total_llm_entities: int = 0

print(f"Starting NER error analysis on {len(data_samples)} samples...")

for i, sample in enumerate(data_samples):
    if 'llm_prediction' not in sample or 'ground_truth' not in sample:
        print(f"Warning: Sample {i+1} is missing 'llm_prediction' or 'ground_truth'. Skipping.")
        continue

    llm_pred = sample['llm_prediction']
    gt = sample['ground_truth']

    # Analyze the single sample
    try:
        sample_results = analyze_ner_errors(llm_pred, gt)
        # Update total results
        total_results.update(sample_results)
        # Update total entity counts
        total_gt_entities += len(gt)
        total_llm_entities += len(llm_pred)
    except Exception as e:
        print(f"Error processing sample {i+1}: {e}")
        # Optionally add more details about the sample causing the error

print("Analysis complete.")


# --- 3. Display Results ---
print("\n--- Overall NER Error Analysis ---")
print(f"Total Samples Analyzed: {len(data_samples)}")
print("-" * 35)
print(f"Total Ground Truth Entities:   {total_gt_entities}")
print(f"Total LLM Predicted Entities:  {total_llm_entities}")
print("-" * 35)
print("Entity Counts:")
print(f"  Correct Matches:      {total_results['correct']}")
print(f"  Entity Type Errors:   {total_results['type']}")
print(f"  Entity Span Errors:   {total_results['span']} (using substring matching)")
print(f"  Missing Entities:     {total_results['missing']}")
print(f"  Spurious Entities:    {total_results['spurious']}")
print("-" * 35)

# --- Performance Metrics (Strict - Type/Span count as FP & FN) ---
# True Positives (TP) = Correct Matches
# False Positives (FP) = Spurious Errors + Type Errors + Span Errors
# False Negatives (FN) = Missing Errors + Type Errors + Span Errors

tp = total_results['correct']
fp = total_results['spurious'] + total_results['type'] + total_results['span']
fn = total_results['missing'] + total_results['type'] + total_results['span']

# Avoid division by zero
precision = tp / (tp + fp) if (tp + fp) > 0 else 0
recall = tp / (tp + fn) if (tp + fn) > 0 else 0
f1_score = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

print("Performance Metrics (Strict Matching):")
print(f"  Precision: {precision:.4f}  (TP / (TP + FP))")
print(f"  Recall:    {recall:.4f}  (TP / (TP + FN))")
print(f"  F1-Score:  {f1_score:.4f}")
print("-" * 35)

Starting NER error analysis on 29 samples...
Analysis complete.

--- Overall NER Error Analysis ---
Total Samples Analyzed: 29
-----------------------------------
Total Ground Truth Entities:   156
Total LLM Predicted Entities:  166
-----------------------------------
Entity Counts:
  Correct Matches:      104
  Entity Type Errors:   30
  Entity Span Errors:   14 (using substring matching)
  Missing Entities:     8
  Spurious Entities:    18
-----------------------------------
Performance Metrics (Strict Matching):
  Precision: 0.6265  (TP / (TP + FP))
  Recall:    0.6667  (TP / (TP + FN))
  F1-Score:  0.6460
-----------------------------------


## 方法1: 分类别去Refiner

In [22]:
import json
import random
import os
from collections import Counter
import collections
from utils import save_jsonl, load_jsonl, load_json, save_json, extract_entities, evaluate_sent, compute_f1, analyze_ner_errors, get_present_ner_error_types
from typing import List, Tuple, Dict, Counter as CounterType

# for i in [1, 5, 25, 50, 100]:
# output/aug/ai/deepseek-chat_r1.json
data_path = "./datasets/aug/ai/deepseek-chat-0-shots-ICL-errors.json"
data = load_json(data_path)
counts = Counter()
for sent in data:
    if "response" not in sent:
        continue
    sent['response'] = sent['response'].replace("Output: ", "")
    pred = extract_entities(sent['response'])
    ref = sent['ners']
    counts = evaluate_sent(ref, pred, counts)
scores_ner = compute_f1(counts["ner_predicted"], counts["ner_gold"], counts["ner_matched"])
# print(f"======= {i} =======")
print(scores_ner)

{'precision': 0.7448979591836735, 'recall': 0.6860902255639098, 'f1': 0.7142857142857143}


In [23]:
data[0].keys()

dict_keys(['idx', 'text', 'ners', 'target', 'sent_id', 'str_words', 'tags_ner', 'response', 'pred', 'errors', 'correct'])

In [ ]:
# 构架一个函数，根据是否存在某种error来构建demos
# 方法1: 修正XML-style annotation
sent = data[0]
one_demo = ""
one_demo += f"Text: {sent['text']}\n"
one_demo += f"Initial Annotation: {sent['response']}\n"
type_error = sent['errors']['type']
if type_error:
    # replace the type error with the correct one
    pre_ann = sent['response']
    for t_error in type_error:
        ent_tag_wrong = f"<entity type=\"{t_error[1]}\">{t_error[0]}</entity>"
        ent_tag = f"<entity type=\"{t_error[3]}\">{t_error[0]}</entity>"
        # replace the wrong entity with the correct one
        pre_ann = pre_ann.replace(ent_tag_wrong, ent_tag)
    one_demo += f"Type Refined Annotation: {pre_ann}\n"
print(one_demo)

# 方法2: 修正Entity List即可
# sent = data[0]
# one_demo = ""
# one_demo += f"Text: {sent['text']}\n"
# one_demo += f"Initial Entity List: {sent['pred']}\n"
# type_error = sent['errors']['type']
# pred_ents = sent['pred']
# # print("Before: ", pred_ents)
# if type_error:
#     for t_error in type_error:
#         error_one = [t_error[0], t_error[1]]
#         target = [t_error[0], t_error[3]]
#         # replace the wrong entity with the correct one
#         for i in range(len(pred_ents)):
#             if pred_ents[i] == error_one:
#                 pred_ents[i] = target
# # print("Type Error Refined Entity List: ", pred_ents)
# one_demo += f"Refined Entity List: {pred_ents}\n"
# print(one_demo)

Text: Popular approaches of opinion-based recommender system utilize various techniques including text mining , information retrieval , sentiment analysis ( see also Multimodal sentiment analysis ) and deep learning X.Y. Feng , H. Zhang , Y.J. Ren , P.H. Shang , Y. Zhu , Y.C. Liang , R.C. Guan , D. Xu , ( 2019 ) , , 21 ( 5 ) : e12957 .
Initial Entity List: [['recommender system', 'product'], ['text mining', 'field'], ['information retrieval', 'task'], ['sentiment analysis', 'task'], ['Multimodal sentiment analysis', 'task'], ['deep learning', 'field'], ['X.Y. Feng', 'researcher'], ['H. Zhang', 'researcher'], ['Y.J. Ren', 'researcher'], ['P.H. Shang', 'researcher'], ['Y. Zhu', 'researcher'], ['Y.C. Liang', 'researcher'], ['R.C. Guan', 'researcher'], ['D. Xu', 'researcher']]
Refined Entity List: [['recommender system', 'product'], ['text mining', 'field'], ['information retrieval', 'task'], ['sentiment analysis', 'task'], ['Multimodal sentiment analysis', 'task'], ['deep learning', 'fiel

In [24]:
def generate_type_error_demos(data_list):
    """
    Args:
        data_list (list of dict): each dict contains keys:
            - 'text': str, the original text
            - 'pred': list of [text_span, type] pairs (initial entity list)
            - 'errors': dict containing 'type': list of type error records
                each type error record is [text_span, wrong_type, _, correct_type]
    
    Returns:
        list of str: each string is a demo following the prompt format
    """
    demos = []

    for idx, sent in enumerate(data_list):
        text = sent['text']
        pred_ents = sent['pred'].copy()  # avoid modifying original
        type_errors = sent.get('errors', {}).get('type', [])

        # Fix type errors
        if type_errors:
            for t_error in type_errors:
                error_one = [t_error[0], t_error[1]]   # [text_span, wrong_type]
                target = [t_error[0], t_error[3]]      # [text_span, correct_type]
                for i in range(len(pred_ents)):
                    if pred_ents[i] == error_one:
                        pred_ents[i] = target

        # Format as one demo
        one_demo = []
        # one_demo.append(f"Example {idx + 1}:")
        one_demo.append(f"Text: {text}")
        one_demo.append(f"Initial Entity List: {sent['pred']}")
        one_demo.append(f"Refined Entity List: {pred_ents}")

        demos.append("\n".join(one_demo))

    return demos

In [25]:
demo_str = generate_type_error_demos(data)

In [30]:
print(demo_str[4])

Text: Variants of the back-propagation algorithm as well as unsupervised methods by Geoff Hinton and colleagues at the University of Toronto can be used to train deep , highly nonlinear neural architectures , { { cite journal
Initial Entity List: [['back-propagation algorithm', 'algorithm'], ['Geoff Hinton', 'researcher'], ['University of Toronto', 'university']]
Refined Entity List: [['back-propagation algorithm', 'algorithm'], ['Geoff Hinton', 'researcher'], ['University of Toronto', 'university']]


## 方法2: Judge + Edit